In [11]:
import duckdb


In [12]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [13]:
df = con.execute("""
                 SELECT  * 
                 FROM (
                 SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row 
                 FROM bronze_z0019
                 WHERE data_ingestao >= '2026-05-03'
                 ) WHERE row = 1
                 """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10004,SERRA,BT50,100,200,z0019_2.csv,2026-05-03 16:37:10.142303,1
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-05-03 16:37:10.142303,1
2,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-05-03 17:02:19.849034,1
3,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-05-03 17:02:19.849034,1
4,10003,PREGO,BT10,100,50,z0019_1.csv,2026-05-03 17:02:19.849034,1


In [16]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final = df_final.rename(columns={"NATBR":"id"})
df_final = df_final.rename(columns={"MAKTX":"nome"})
df_final = df_final.rename(columns={"WERKS":"categoria"})
df_final = df_final.rename(columns={"MAINS":"fornecedor"})
df_final = df_final.rename(columns={"LABST":"preco"})

df_final.head(10)

,id,nome,categoria,fornecedor,preco
0,10004,SERRA,BT50,100,200
1,10005,MACHADO,BT50,100,100
2,10001,PARAFUSO,BT10,100,100
3,10002,MARTELO,BT50,100,1500
4,10003,PREGO,BT10,100,50


In [17]:
df_final.dtypes

id            object
nome          object
categoria     object
fornecedor    object
preco         object
dtype: object

In [ ]:
df2 = df_final
df2 = df2.astype(
    {
        'id': int,
        'nome': str,
        'categoria': str,
        'fornecedor': int,
        'preco': float
})
df2.dtypes


id              int64
nome           object
categoria      object
fornecedor      int64
preco         float64
dtype: object

In [21]:
con.execute("""
            CREATE TABLE IF NOT EXISTS produtos (
            id BIGINT,
            nome TEXT,
            categoria TEXT,
            fornecedor BIGINT,
            valor FLOAT
            )
            
           """)

In [22]:
df2.head(10)

,id,nome,categoria,fornecedor,preco
0,10004,SERRA,BT50,100,200.0
1,10005,MACHADO,BT50,100,100.0
2,10001,PARAFUSO,BT10,100,100.0
3,10002,MARTELO,BT50,100,1500.0
4,10003,PREGO,BT10,100,50.0


In [24]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [25]:
df_resultado = con.execute("SELECT * FROM produtos").fetchdf()
df_resultado.head(10)

,id,nome,categoria,fornecedor,valor
0,10004,SERRA,BT50,100,200.0
1,10005,MACHADO,BT50,100,100.0
2,10001,PARAFUSO,BT10,100,100.0
3,10002,MARTELO,BT50,100,1500.0
4,10003,PREGO,BT10,100,50.0


In [26]:
con.close()